In [0]:
table_name = "dim_members"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_members")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, upper, trim, initcap, lower, regexp_replace, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Standardize Text
df = df.withColumn("member_id", upper(trim(col("member_id"))))
df = df.withColumn("policy_number", upper(trim(col("policy_number"))))
df = df.withColumn("first_name", initcap(trim(col("first_name"))))
df = df.withColumn("last_name", initcap(trim(col("last_name"))))
df = df.withColumn("full_name", initcap(trim(col("full_name"))))
df = df.withColumn("gender", upper(trim(col("gender"))))
df = df.withColumn("pan_number", upper(trim(col("pan_number"))))
df = df.withColumn("state", upper(trim(col("state"))))
df = df.withColumn("city", upper(trim(col("city"))))
df = df.withColumn("employment_status", upper(trim(col("employment_status"))))
df = df.withColumn("chronic_conditions", upper(trim(col("chronic_conditions"))))
df = df.withColumn("member_status", upper(trim(col("member_status"))))
df = df.withColumn("language_preference", initcap(trim(col("language_preference"))))
df = df.withColumn("contact_preference", initcap(trim(col("contact_preference"))))
df = df.withColumn("email", lower(trim(col("email"))))

# 2. Clean Phone (Remove non-numeric)
df = df.withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))

# 3. Cleanse Invalid Formats
# Aadhar: Keep only if length 12 and numeric
df = df.withColumn("aadhar_number", when(trim(col("aadhar_number")).rlike("^[0-9]{12}$"), trim(col("aadhar_number"))).otherwise(None))
# Pincode: Keep only if length 6 and numeric
df = df.withColumn("pincode", when(trim(col("pincode")).rlike("^[0-9]{6}$"), trim(col("pincode"))).otherwise(None))

# 4. Safe Cast Dates (Removed strict format string so Spark auto-handles timestamps)
for date_col in ["dob", "enrollment_date", "termination_date", "last_claims_date", "record_created_date", "record_modified_date"]:
    df = df.withColumn(date_col, to_date(trim(col(date_col))))

# 5. Safe Cast Integers
for int_col in ["age", "claims_history_count"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))

# 6. Precision Optimization (Financials & Scores) - SAFE CAST IMPLEMENTATION
# If the value is not a number (like "Unlimited"), it becomes NULL instead of crashing the pipeline.
for dec_col in ["sum_insured", "deductible_amount", "room_rent_limit"]:
    # Regex checks if the trimmed string is a valid number (with optional negative sign and decimals)
    is_valid_decimal = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid_decimal, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    df = df.withColumn(dec_col, spark_abs(col(dec_col))) # If < 0, ABS() (NULLs are ignored by abs)

# For percentages and small scores
is_valid_percent = trim(col("co_pay_percentage")).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
df = df.withColumn("co_pay_percentage", when(is_valid_percent, trim(col("co_pay_percentage")).cast("decimal(5,2)")).otherwise(None))

is_valid_score = trim(col("risk_score")).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
df = df.withColumn("risk_score", when(is_valid_score, trim(col("risk_score")).cast("decimal(5,2)")).otherwise(None))
df = df.withColumn("risk_score", when(col("risk_score") == 99999, None).otherwise(col("risk_score"))) # 99999 -> NULL

# 7. Boolean Mapping
for bool_col in ["tobacco_user", "disability_status"]:
    df = df.withColumn(bool_col, 
                       when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True)
                       .when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False)
                       .otherwise(None).cast("boolean"))

# 8. Deduplication (Keep latest ingested record per member_id)
dedup_window = Window.partitionBy("member_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window))
df = df.filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped")

# 9. Drop audit columns and source surrogate keys
df = df.drop("_ingested_at", "_source_file", "member_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")